In [61]:
import pandas as pd
from linearmodels.panel import PanelOLS
import numpy as np

We begin by performing our baseline regression and inspecting our results.

In [62]:
baseline_reg = pd.read_csv("csv_data/BASELINE1.csv")
baseline_reg.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel', 're', 'volatility', 'roa',
       'adjusted_roa', 'firm_size', 'equity_capital', 'charter_value',
       'retained_earnings', 'rd_intensity', 'ceo_age', 'ceo_gender',
       'masters_factor', 'md_factor', 'phd_factor', 'mba_factor', 'ug_factor'],
      dtype='object')

In [63]:
print(baseline_reg['gender'].value_counts(dropna=False))
print(baseline_reg['gender'].unique())

gender
M      958
F       56
NaN     18
Name: count, dtype: int64
['F' 'M' nan]


In [64]:
baseline_reg['CEO_gender'] = baseline_reg['gender'].map({'M': 0, 'F': 1})
print(baseline_reg['CEO_gender'].value_counts(dropna=False))

CEO_gender
0.0    958
1.0     56
NaN     18
Name: count, dtype: int64


In [70]:
reg_vars = ['adjusted_roa', 'roa', 'ug_factor', 'mba_factor', 'phd_factor', 
            'md_factor', 'masters_factor', 'firm_size', 'equity_capital', 
            'charter_value', 'retained_earnings', 'rd_intensity', 
            'volatility', 'ceo_age', 'CEO_gender']

# Drop NaNs on regression variables only
baseline_reg_clean = baseline_reg.dropna(subset=reg_vars)
print(baseline_reg_clean.shape)
print(baseline_reg_clean['CEO_gender'].value_counts(dropna=False))

(943, 64)
CEO_gender
0.0    891
1.0     52
Name: count, dtype: int64


In [71]:
print(baseline_reg_clean['gender'].value_counts(dropna=False))

gender
M    891
F     52
Name: count, dtype: int64


In [72]:
panel_data = baseline_reg_clean.set_index(['gvkey', 'fyear'])

TBU

In [74]:
# Simplest possible model first
model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_data,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.2661
Estimator:                   PanelOLS   R-squared (Between):              0.0567
No. Observations:                 943   R-squared (Within):               0.2681
Date:                Sat, Apr 25 2026   R-squared (Overall):              0.0529
Time:                        11:39:14   Log-likelihood                    1310.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      23.268
Entities:                         151   P-value                           0.0000
Avg Obs:                       6.2450   Distribution:                  F(12,770)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             8.9643
                            

In [75]:
biotech = baseline_reg_clean[baseline_reg_clean['industry'] == 'Biotech']
semis = baseline_reg_clean[baseline_reg_clean['industry'] == 'Semiconductors/Hardware']
software = baseline_reg_clean[baseline_reg_clean['industry'] == 'Software']

In [79]:
panel_bio = biotech.set_index(['gvkey', 'fyear'])

model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_bio,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.4473
Estimator:                   PanelOLS   R-squared (Between):             -0.2671
No. Observations:                 340   R-squared (Within):               0.4440
Date:                Sat, Apr 25 2026   R-squared (Overall):              0.0478
Time:                        11:42:29   Log-likelihood                    494.82
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      17.603
Entities:                          57   P-value                           0.0000
Avg Obs:                       5.9649   Distribution:                  F(12,261)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             14.122
                            

In [ ]:
panel_semis = semis.set_index(['gvkey', 'fyear'])

model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_semis,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.2301
Estimator:                   PanelOLS   R-squared (Between):             -0.0955
No. Observations:                 389   R-squared (Within):               0.2178
Date:                Sat, Apr 25 2026   R-squared (Overall):              0.0221
Time:                        11:40:42   Log-likelihood                    597.15
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.7720
Entities:                          55   P-value                           0.0000
Avg Obs:                       7.0727   Distribution:                  F(12,312)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             15.473
                            

In [ ]:
panel_software = software.set_index(['gvkey', 'fyear'])

model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_software,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.3797
Estimator:                   PanelOLS   R-squared (Between):             -0.0561
No. Observations:                 214   R-squared (Within):               0.3610
Date:                Sat, Apr 25 2026   R-squared (Overall):             -0.2704
Time:                        11:40:56   Log-likelihood                    306.13
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.8050
Entities:                          39   P-value                           0.0000
Avg Obs:                       5.4872   Distribution:                  F(12,153)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             7.6422
                            